In [ ]:
!pip install transformers torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.0 MB/s eta 0:00:00


In [ ]:
import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PROBES_PATH = "/content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_probes.json"
RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(PROBES_PATH) as f:
    PROBE_ROWS = json.load(f)
print(f"Loaded {len(PROBE_ROWS)} probe rows.")

Loaded 96 probe rows.


In [ ]:
def forced_choice(model, tokenizer, prompt, option_a, option_b):
    """ two-full-pass comparison ; not used in the main loop."""
    def score_continuation(continuation):
        full_ids = tokenizer(prompt + continuation, return_tensors="pt").input_ids.to(model.device)
        prompt_len = tokenizer(prompt, return_tensors="pt").input_ids.shape[1]
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        log_probs = torch.log_softmax(logits, dim=-1)
        total = 0.0
        for i in range(prompt_len, full_ids.shape[1]):
            total += log_probs[i - 1, full_ids[0, i]].item()
        return total

    score_a = score_continuation(option_a)
    score_b = score_continuation(option_b)
    winner = option_a if score_a > score_b else option_b
    return winner, {"option_a": option_a, "score_a": score_a,
                     "option_b": option_b, "score_b": score_b}


def mcq_choice(model, tokenizer, prompt, letter_map):
    """
    Score a lettered MCQ probe with one forward pass on the prompt,
    reading next-token log-probs of ' A' vs ' B' directly.
    letter_map: {"A": "meaning", "B": "meaning"}, as stored in
    redundancy_probes.json alongside each mcq prompt.
    Returns: (winner_meaning, winner_letter, scores_dict)
    """

    token_a_ids = tokenizer(" A", add_special_tokens=False).input_ids
    token_b_ids = tokenizer(" B", add_special_tokens=False).input_ids
    if len(token_a_ids) != 1 or len(token_b_ids) != 1:
        winner_letter_str, scores = forced_choice(model, tokenizer, prompt, " A", " B")
        winner_letter = winner_letter_str.strip()
        return letter_map[winner_letter], winner_letter, scores
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        next_token_logits = model(input_ids).logits[0, -1]
    log_probs = torch.log_softmax(next_token_logits, dim=-1)
    score_a = log_probs[token_a_ids[0]].item()
    score_b = log_probs[token_b_ids[0]].item()
    winner_letter = "A" if score_a > score_b else "B"
    scores = {"option_a": " A", "score_a": score_a, "option_b": " B", "score_b": score_b}
    return letter_map[winner_letter], winner_letter, scores

In [ ]:
CHECKPOINTS = {
    "pretrained": {"name": "allenai/OLMo-2-0425-1B", "format": "completion"},
    "sft":        {"name": "allenai/OLMo-2-0425-1B-SFT", "format": "chat"},
    "dpo":        {"name": "allenai/OLMo-2-0425-1B-DPO", "format": "chat"},
}


def load_model(checkpoint_name):
    print(f"Loading {checkpoint_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_name,
        torch_dtype=torch.float32,
        load_in_8bit=True,   # keeps a 7B model within a T4's memory
        device_map="auto",
    ).to(device)

    print(f"  loaded on: {device}")
    return model, tokenizer


def format_for_scoring(prompt, fmt, tokenizer):
    """Chat template for SFT/DPO checkpoints; pass-through for the
    pretrained/completion checkpoint (MCQ's (A)/(B) structure alone
    cues the expected answer shape, no few-shot example needed)."""
    if fmt == "chat":
        messages = [{"role": "user", "content": prompt}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return prompt

In [ ]:
STEPS = ["script_knowledge", "redundancy_id", "inference_derivation"]
RELIABLE_STEPS = {"script_knowledge", "inference_derivation"}  # redundancy_id parked (known bias, see project notes)


def already_done(results_path):
    done = set()
    if os.path.exists(results_path):
        with open(results_path) as f:
            for line in f:
                r = json.loads(line)
                done.add((r["item_id"], r["condition"], r["step"]))
    return done


def run_checkpoint(stage_name, rows):
    checkpoint_name = CHECKPOINTS[stage_name]["name"]
    fmt = CHECKPOINTS[stage_name]["format"]
    results_path = os.path.join(RESULTS_DIR, f"{stage_name}.jsonl")

    done = already_done(results_path)
    total_calls = len(rows) * len(STEPS)
    if len(done) == total_calls:
        print(f"[{stage_name}] already complete ({total_calls} calls) -- skipping.")
        return

    model, tokenizer = load_model(checkpoint_name)

    with open(results_path, "a") as f:
        call_count = 0
        for row in rows:
            for step in STEPS:
                key = (row["item_id"], row["condition"], step)
                if key in done:
                    continue

                mcq_prompt = row["probes"][step]["mcq"]
                letter_map = row["probes"][step]["mcq_letter_map"]

                try:
                    formatted = format_for_scoring(mcq_prompt, fmt, tokenizer)
                    winner_meaning, winner_letter, scores = mcq_choice(
                        model, tokenizer, formatted, letter_map
                    )
                    error = None
                except Exception as e:#to catch failures if any
                    winner_meaning, winner_letter, scores = None, None, None
                    error = str(e)

                record = {
                    "stage": stage_name,
                    "checkpoint": checkpoint_name,
                    "item_id": row["item_id"],
                    "character": row["character"],
                    "condition": row["condition"],
                    "step": step,
                    "target_property": row["target_property"],
                    "wonky_property": row["wonky_property"],
                    "mcq_prompt": mcq_prompt,
                    "letter_map": letter_map,
                    "winner_meaning": winner_meaning,
                    "winner_letter": winner_letter,
                    "scores": scores,
                    "reliable": step in RELIABLE_STEPS,
                    "error": error,
                }
                f.write(json.dumps(record) + "\n")
                f.flush()

                call_count += 1
                if call_count % 20 == 0:
                    print(f"[{stage_name}] {call_count}/{total_calls - len(done)} new calls done")

    print(f"[{stage_name}] finished -> {results_path}")
    del model
    torch.cuda.empty_cache()

In [1]:
#since the output showed infinte numebr of warnings popping up, not allowing to see the generated results clearly.
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

In [ ]:
#make sure the gpu is connected before running the checkpoitn

import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [ ]:
CHECKPOINT_TO_RUN = "pretrained"  # change to "sft" or "pretrained" for the other runs
run_checkpoint(CHECKPOINT_TO_RUN, PROBE_ROWS)




Loading allenai/OLMo-2-1124-7B ...


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

[pretrained] 20/288 new calls done
[pretrained] 40/288 new calls done
[pretrained] 60/288 new calls done
[pretrained] 80/288 new calls done
[pretrained] 100/288 new calls done
[pretrained] 120/288 new calls done
[pretrained] 140/288 new calls done
[pretrained] 160/288 new calls done
[pretrained] 180/288 new calls done
[pretrained] 200/288 new calls done
[pretrained] 220/288 new calls done
[pretrained] 240/288 new calls done
[pretrained] 260/288 new calls done
[pretrained] 280/288 new calls done
[pretrained] finished -> /content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/pretrained.jsonl
